In [1]:
import os

In [2]:
%pwd

'c:\\Users\\ganni\\OneDrive\\Desktop\\End-to-End-Machine-Learning-Project-with-ML-FLOW\\research'

In [3]:

os.chdir("../")

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [5]:
from mlproject.constants import *
from mlproject.utils.common import read_yaml, create_directories

In [10]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath: Path = CONFIG_FILE_PATH,
            params_filepath: Path = PARAMS_FILE_PATH,
            schema_filepath: Path = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([Path(self.config.artifacts_root)])

    def get_data_ingestion_config(self) -> DataIngestionConfig:

        config = self.config.data_ingestion

        create_directories([Path(config.root_dir)])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )

        return data_ingestion_config

In [7]:
import os
import urllib.request as request
import zipfile
from mlproject import logger
from mlproject.utils.common import get_size

In [8]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_data(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} downloaded with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def unzip_and_clean(self):
        with zipfile.ZipFile(file=self.config.local_data_file, mode="r") as zip_ref:
            zip_ref.extractall(path=self.config.unzip_dir)
        logger.info(f"Unzipped the file in directory: {self.config.unzip_dir}")

In [13]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_data()
    data_ingestion.unzip_and_clean()

except Exception as e:
    raise e

[2026-09-24 21:22:10,385: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-24 21:22:10,386: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-24 21:22:10,387: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-09-24 21:22:10,389: INFO: common: created directory at: artifacts]
[2026-09-24 21:22:10,390: INFO: common: created directory at: artifacts\data_ingestion]
[2026-09-24 21:22:10,392: INFO: 3938641823: File already exists of size: ~ 23 KB]
[2026-09-24 21:22:10,403: INFO: 3938641823: Unzipped the file in directory: artifacts\data_ingestion]
